# Linear Regression

Linear regression predicts a **continuous number** by fitting a straight line (or, with many features, a flat hyperplane) as close as possible to the data. With one feature it is just the familiar line

$$\hat{y} = m\,x + b$$

and with $n$ features it generalizes to a weighted sum plus an intercept, written compactly in matrix form as

$$\hat{y} = Xw + b$$

where $X \in \mathbb{R}^{m \times n}$ is the feature matrix ($m$ rows/samples, $n$ columns/features), $w \in \mathbb{R}^{n}$ is the vector of slopes (one per feature), and $b$ is a single intercept. It is the simplest and most interpretable regression model, which makes it the ideal first algorithm.

**Topics covered in this notebook**

1. Intuition
2. Training on a dataset
3. Inspecting the model
4. When to use it

## 1. Intuition

The model learns a slope for each feature plus one shared intercept so that the fitted line sits as close as possible to all the points. "As close as possible" is made precise by the **least-squares objective**: choose $w$ and $b$ that minimize the sum of squared vertical gaps (residuals) between the true values $y_i$ and the predictions $\hat{y}_i$:

$$\min_{w,\,b} \; \sum_{i=1}^{m} \big(y_i - \hat{y}_i\big)^2 = \min_{w,\,b} \; \sum_{i=1}^{m} \big(y_i - (x_i^{\top} w + b)\big)^2$$

Squaring the residuals (instead of taking absolute values) makes big misses hurt disproportionately and gives a smooth objective with a single closed-form optimum, which is exactly what scikit-learn's `LinearRegression` solves under the hood.

Below is a tiny one-feature fit so we can see the moving parts: the learned slope `m`, the intercept `b`, and a prediction for a brand-new `x`.

In [ ]:
import matplotlib.pyplot as plt          # plotting only
import numpy as np                       # arrays and shaping
from sklearn.linear_model import LinearRegression  # the ordinary least-squares model

# 1. Define a tiny toy dataset and fit the model.
# scikit-learn expects the feature matrix X to be 2-D with shape (n_samples, n_features).
# We have 5 samples and 1 feature, so reshape the flat array (5,) -> (5, 1).
x = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)   # shape (5, 1): 5 samples, 1 feature
y = np.array([2, 4, 5, 4, 6])                  # shape (5,):  the continuous target

# .fit() solves the least-squares problem above and stores the best slope + intercept.
line = LinearRegression().fit(x, y)

In [ ]:
# Read back the two learned numbers.
# coef_ holds one slope per feature (here just one), intercept_ is the single bias term b.
print(f"slope (m): {line.coef_[0]:.2f}")
print(f"intercept (b): {line.intercept_:.2f}")

# Predict for a new input x=6. We pass it as a 2-D array [[6]] (shape (1, 1)) to match
# the (n_samples, n_features) shape used during fit; predict() returns an array, so [0]
# grabs the single scalar prediction.
print("prediction at x=6:", round(line.predict(np.array([[6]]))[0], 2))

In [ ]:
# 2. Simplest possible visualization of the fit.
# Create this cell's OWN figure/axes so the plot never depends on leftover state
# from an earlier cell (robust, re-runnable in any order).
fig, ax = plt.subplots(figsize=(6, 4))

ax.scatter(x, y, color="green", label="Data points")            # the raw observations
ax.plot(x, line.predict(x), color="orange", label="Fit line")   # the least-squares line

# Highlight the out-of-sample prediction at x=6 with a red X.
ax.scatter(6, line.predict(np.array([[6]])), color="red", marker="x", s=100,
           label="Prediction (x=6)")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("One-feature least-squares fit")
ax.legend()
plt.show()

## 2. Training on a Dataset

The real workflow: load data, **split** it into a training set and a held-out test set, `fit` on the training set only, then measure error on data the model has never seen. Evaluating on held-out data is what tells you whether the model *generalizes* rather than just memorizing.

We use scikit-learn's built-in **diabetes** dataset (no download): 10 standardized physiological features (age, BMI, blood pressure, six blood-serum measurements, etc.) mapping to a continuous disease-progression score one year later.

We report two complementary metrics:

- **$R^2$ (coefficient of determination)** — the fraction of the target's variance the model explains. $1.0$ is a perfect fit, $0$ is no better than always predicting the mean, and it can go negative for a model worse than the mean:

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

- **RMSE (root mean squared error)** — the typical prediction error, expressed in the *same units as the target* (so it is directly interpretable):

$$\text{RMSE} = \sqrt{\frac{1}{m}\sum_{i=1}^{m} (y_i - \hat{y}_i)^2}$$

In [ ]:
from sklearn.datasets import load_diabetes            # built-in regression dataset (offline)
from sklearn.model_selection import train_test_split  # random train/test split
from sklearn.metrics import mean_squared_error, r2_score  # the two evaluation metrics

# Load features X (shape (442, 10)) and the continuous target y (shape (442,)).
X, y = load_diabetes(return_X_y=True)

# Hold out 20% of the rows for testing. random_state fixes the shuffle so the split
# (and therefore every number below) is reproducible across runs.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit ordinary least squares on the TRAINING data only, then predict the held-out rows.
model = LinearRegression()
model.fit(X_train, y_train)          # solves min ||X_train w + b - y_train||^2
preds = model.predict(X_test)        # predictions for data the model never saw, shape (89,)

# R^2: fraction of variance explained on the test set (closer to 1.0 is better).
print("R^2 :", round(r2_score(y_test, preds), 3))
# RMSE: take the square root of MSE ourselves (** 0.5) to stay version-agnostic and
# avoid any deprecated 'squared=' keyword. Units match the disease-progression score.
print("RMSE:", round(mean_squared_error(y_test, preds) ** 0.5, 2))

In [ ]:
# Predicted-vs-actual scatter: a quick visual read on fit quality.
# Perfect predictions would land exactly on the dashed diagonal (predicted == actual).
# Points scattered around it show the model captures the trend but not every case.
fig, ax = plt.subplots(figsize=(6, 5))

ax.scatter(y_test, preds, color="purple", alpha=0.6, label="Test predictions")

# Draw the y = x reference line spanning the observed range of the target.
lo, hi = y_test.min(), y_test.max()
ax.plot([lo, hi], [lo, hi], "r--", label="Perfect prediction")

ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.set_title("Predicted vs actual (test set)")
ax.legend()
plt.show()

In [ ]:
# Peek at the first 5 predictions next to the true values to feel the error scale.
# They are in the same ballpark but clearly not exact - consistent with R^2 ~= 0.45.
print("predicted: ", preds[:5].round(2))
print("actual   : ", y_test[:5])

## 3. Inspecting the Model

Linear regression is prized for being **interpretable**. Because the prediction is a plain weighted sum $\hat{y} = w_1 x_1 + \dots + w_n x_n + b$, each coefficient $w_j$ has a direct meaning: *holding the other features fixed, increasing feature $j$ by one unit changes the prediction by $w_j$.* The intercept $b$ is the baseline prediction when every feature is zero (here, at each feature's mean, since the diabetes features are standardized).

In [ ]:
# The intercept b: the model's baseline output.
print("intercept:", round(model.intercept_, 2))

# One coefficient per input feature -> 10 for the diabetes dataset.
print("number of coefficients:", len(model.coef_))

# The first 3 slopes. Sign matters: a positive coef pushes progression up as that
# feature rises, a negative coef pushes it down; larger magnitude = stronger effect.
print("first 3 coefficients:", model.coef_[:3].round(2))

## 4. When to Use It

- Predicting a **continuous** value (price, score, demand, progression).
- You want a fast, transparent **baseline** whose coefficients you can explain to stakeholders.
- The relationship is roughly linear; standardize/scale features so coefficients are comparable, and watch for **outliers**, which the squared-error objective lets pull the line hard.
- For predicting a category (yes/no), reach for **logistic regression** instead - it reuses this exact linear score $Xw + b$ but squashes it into a probability.

To make the fit visible on a page, the cell below fits a **BMI-only** model. BMI (column index 2) is the single most influential feature, so its one-feature trend is easy to read - while remembering the full 10-feature model above is what we actually evaluate.

In [ ]:
# Show the fit on ONE feature (BMI) so it is easy to read on a 2-D plot.
# The full model uses all 10 features, so plotting its predictions against BMI alone
# would zigzag (the other 9 features vary too). We fit a SEPARATE BMI-only model purely
# to visualise that single feature's trend.
bmi_train = X_train[:, 2].reshape(-1, 1)   # column 2 = BMI, reshaped to (n, 1) for sklearn
bmi_test  = X_test[:, 2].reshape(-1, 1)

# Fit one-feature least squares: progression ~ BMI.
bmi_model = LinearRegression().fit(bmi_train, y_train)

# Sort the test x-values so the fitted line is drawn smoothly left-to-right
# (matplotlib connects points in array order; unsorted x would scribble).
order = bmi_test[:, 0].argsort()

# This cell's own figure/axes -> no dependence on any previous plot's state.
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(bmi_test[:, 0], y_test, color="green", alpha=0.6, label="Actual")
ax.plot(bmi_test[order, 0], bmi_model.predict(bmi_test[order]),
        color="red", label="BMI-only fit")
ax.set_xlabel("BMI (standardized)")
ax.set_ylabel("Disease progression")
ax.set_title("Diabetes: BMI vs Progression")
ax.legend()
plt.show()